# Segmentación por **crecimiento de regiones** con validación cruzada de **K = 5** y test fijo

Aplica **crecimiento de regiones (region growing)** a todas las imágenes de
`DATASET_FINAL2.mat`.

* validación cruzada **estratificada por tipo de lesión**, K = 5 pliegues
* las imágenes de índice **23–27** (con imagen registrada) son **siempre test** y nunca entran en entrenamiento ni en validación
* el resto de imágenes rota: en cada pliegue, 80 % train+val (reparto interno 80/20) y
  20 % test, que se suma al test fijo
* métricas reportadas como **media** entre pliegues, desglosadas en test total,
  solo rotatorias y solo fijas

El crecimiento de regiones no aprende parámetros: lo que se selecciona en cada pliegue es la
**configuración del crecimiento** (mapa de siembra, espacio de color, regla de agregación y
tolerancia), igual que en el cuaderno de K-means se seleccionaba `(espacio, k, criterio)`.


### 1. Montar Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### 2. Parámetros

In [5]:
RUTA_MAT      = "/C:/Users/josem/Desktop/DATASET_FINAL2.pdf"   # ruta al .mat (v7.3)

UMBRAL_DICE   = 0.9      # solo se muestran las imágenes que superen este Dice
MAX_MOSTRAR   = 5        # tope de filas en la figura (None = todas)
POSTPROCESO   = True     # limpieza morfológica + mayor componente conexa
RELLENAR      = True     # rellena los huecos negros dentro de la lesión
GUARDAR       = True     # guardar la figura en CARPETA_SALIDA

REDIMENSIONAR = True     # 256x256, como el resto del pipeline del TFG.
IMG_SIZE      = 256      # Ponlo a False para trabajar a resolución nativa

# --- validación cruzada ---
N_SPLITS      = 5        # K = 5 pliegues
VAL_FRACTION  = 0.20     # 20 % del 80 % restante = 16 % del total
RANDOM_STATE  = 42       # semilla fija -> pliegues reproducibles

# --- test fijo -------------------------------------------------------------
# Imágenes con imagen registrada del dataset: SIEMPRE son test, en todos los
# pliegues, y NUNCA entran en entrenamiento ni en validación.
INDICES_TEST_FIJOS = list(range(23, 28))   # 23, 24, 25, 26, 27

# --- crecimiento de regiones -----------------------------------------------
SUAVIZADO     = 5        # tamaño del filtro gaussiano previo
PESO_CENTRO   = 0.30     # peso del prior de centralidad en los mapas "*_c"
P_SEMILLA     = 99       # percentil del mapa que define la semilla inicial (~1 % de píxeles)
CONECTIVIDAD  = 4        # 4 u 8 vecinos al expandir la región
MAX_ITER      = 400      # tope de iteraciones de crecimiento
AREA_MAXIMA   = 0.90     # si la región supera este área se corta: se ha desbordado
SIGMA_MIN     = 0.10     # suelo de la desviación de la región (evita que la regla
                         # "estadistica" se bloquee cuando la semilla es casi uniforme)

# Espacio de búsqueda de la configuración: (mapa, espacio, regla, tol).
#   mapa    -> mapa escalar del que sale la SEMILLA
#   espacio -> características de color con las que se mide la HOMOGENEIDAD
#   regla   -> condición para admitir un píxel vecino en la región
#   tol     -> tolerancia de esa condición (en unidades tipificadas)
CONFIGS_CANDIDATAS = [
    ("a_lab",      "lab_ab", "media",       1.2),   # distancia a la media de la región
    ("a_lab",      "lab_ab", "media",       1.8),
    ("a_lab",      "lab_ab", "estadistica", 2.0),   # k sigmas de la propia región
    ("a_lab",      "lab_ab", "estadistica", 2.8),
    ("a_lab",      "lab_ab", "semilla",     1.5),   # distancia a la media de la semilla
    ("a_lab",      "lab",    "media",       1.5),   # incluye luminancia
    ("a_lab_c",    "lab_ab", "semilla",     2.0),   # siembra con prior de centralidad
    ("a_menos_b",  "lab_ab", "media",       1.5),
    ("a_menos_b",  "lab_ab", "estadistica", 2.5),
    ("saturacion", "lab_ab", "semilla",     1.8),
    ("rojez",      "rgb",    "media",       1.5),
    ("oscuridad",  "gris",   "estadistica", 2.5),
]

### 3. Importaciones

In [6]:
import os
import h5py
import cv2
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import binary_fill_holes
from sklearn.model_selection import StratifiedKFold, train_test_split
from collections import Counter

np.random.seed(RANDOM_STATE)
cv2.setRNGSeed(RANDOM_STATE)

### 4. Lectura del dataset

Se cargan **todas** las entradas (no una selección), porque la validación cruzada necesita el
conjunto completo.

In [7]:
f  = h5py.File(RUTA_MAT, "r")
DS = f["DATASET_UNIDO"]

def leer_entrada(i):
    c = [f[r] for r in f[DS[i, 0]][()].ravel()]
    nombre = "".join(chr(x) for x in c[0][()].ravel())
    tipo   = "".join(chr(x) for x in c[2][()].ravel())
    mask   = (c[1][()].T > 0).astype(np.uint8)                          # (H, W) binaria
    img    = (np.transpose(c[3][()], (2, 1, 0)) * 255).clip(0, 255).astype(np.uint8)
    return nombre, tipo, img, mask


N = DS.shape[0]
nombres, tipos, imagenes, mascaras = [], [], [], []

for i in range(N):
    nombre, tipo, img, gt = leer_entrada(i)
    if REDIMENSIONAR:
        # área para la imagen, vecino más próximo para la máscara,
        # que así conserva su carácter estrictamente binario
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
        gt  = cv2.resize(gt,  (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
    nombres.append(nombre)
    tipos.append(tipo)
    imagenes.append(img)
    mascaras.append((gt > 0).astype(np.uint8))

etiquetas = np.array(tipos)
print(f"{N} imágenes cargadas. ")

130 imágenes cargadas. 


### 5. Segmentación por crecimiento de regiones

La idea es la clásica: se parte de una **semilla** dentro de la lesión y se van anexionando
píxeles vecinos mientras cumplan un **criterio de homogeneidad**, hasta que ninguno lo
cumple. Cuatro piezas:

1. **Mapa de siembra** — un escalar por píxel que mide cuánto "parece lesión": `a_lab`
   (canal *a* de LAB), `a_menos_b`, `saturacion`, `rojez` u `oscuridad`, normalizado entre
   sus percentiles 1 y 99. Los mapas terminados en `_c` mezclan un **prior de centralidad**
   con peso `PESO_CENTRO`, el equivalente de lo que hacía `PESO_XY` en K-means. La semilla
   es la mayor componente conexa por encima del percentil `P_SEMILLA` (~1 % de la imagen):
   varios píxeles, no uno solo, para que la media inicial no dependa de un píxel de ruido.
2. **Espacio de homogeneidad** — el color con el que se compara: `lab_ab`, `lab`, `rgb` o
   `gris`, **tipificado** por canal (media 0, desviación 1) para que ninguno domine la
   distancia euclídea.
3. **Regla de agregación** — un vecino entra en la región si:
   * `media`: su distancia a la **media actual de la región** ≤ `tol` (la referencia se
     recalcula en cada iteración, así que la región se adapta a degradados suaves);
   * `estadistica`: su distancia ≤ `tol` × **desviación de la región** (criterio de *k*
     sigmas, se autoajusta a lesiones más o menos homogéneas);
   * `semilla`: su distancia a la media de la **semilla inicial**, que nunca cambia (el
     criterio más estricto, equivale a un *flood fill* de rango fijo).
4. **Parada** — cuando ninguna candidata cumple el criterio, al llegar a `MAX_ITER`, o si la
   región supera `AREA_MAXIMA` de la imagen, señal de que se ha **desbordado** hacia la piel;
   en ese caso se devuelve la última región válida.

La expansión está vectorizada: en vez de una cola de píxeles, cada iteración dilata la región
y evalúa de golpe todo su borde con NumPy. El criterio se calcula **solo sobre los píxeles
del borde**, y la media y la desviación de la región se actualizan de forma **incremental**;
el resultado es idéntico al de recorrer toda la imagen en cada paso, pero unas 15 veces
más rápido.


In [8]:
EPS = 1e-8

def _centralidad(H, W):
    """Mapa 1 en el centro, 0 en el borde. Prior suave de posición."""
    yy, xx = np.mgrid[0:H, 0:W]
    d = np.sqrt(((xx - (W - 1) / 2) / (W / 2)) ** 2 +
                ((yy - (H - 1) / 2) / (H / 2)) ** 2)
    return (1.0 - np.clip(d, 0, 1)).astype(np.float32)


def _norm01(m):
    """Normaliza entre percentiles 1 y 99: robusto frente a brillos y valores extremos."""
    m = m.astype(np.float32)
    lo, hi = float(np.percentile(m, 1)), float(np.percentile(m, 99))
    return np.clip((m - lo) / (hi - lo + EPS), 0, 1).astype(np.float32)


def _mapa_siembra(img, mapa):
    """Mapa escalar (H, W) en [0, 1] con la 'lesionalidad' de cada píxel."""
    usar_centro = mapa.endswith("_c")
    base = mapa[:-2] if usar_centro else mapa

    suave = cv2.GaussianBlur(img, (SUAVIZADO, SUAVIZADO), 0)
    lab   = cv2.cvtColor(suave, cv2.COLOR_RGB2LAB).astype(np.float32)

    if base == "a_lab":
        m = lab[:, :, 1]
    elif base == "a_menos_b":
        m = lab[:, :, 1] - lab[:, :, 2]
    elif base == "saturacion":
        m = cv2.cvtColor(suave, cv2.COLOR_RGB2HSV)[:, :, 1].astype(np.float32)
    elif base == "oscuridad":
        m = -cv2.cvtColor(suave, cv2.COLOR_RGB2GRAY).astype(np.float32)
    elif base == "rojez":
        r = suave[:, :, 0].astype(np.float32)
        g = suave[:, :, 1].astype(np.float32)
        b = suave[:, :, 2].astype(np.float32)
        m = r - 0.5 * (g + b)
    else:
        raise ValueError(f"mapa desconocido: {mapa}")

    m = _norm01(m)
    if usar_centro:
        H, W = m.shape
        m = (1 - PESO_CENTRO) * m + PESO_CENTRO * _centralidad(H, W)
    return m.astype(np.float32)


def _caracteristicas(img, espacio):
    """Imagen (H, W, D) de características tipificadas por canal.

    La tipificación es la misma idea que en K-means: sin ella, un canal con más rango
    (la luminancia frente a la crominancia) dominaría la distancia euclídea y la
    tolerancia `tol` no significaría lo mismo de una imagen a otra."""
    suave = cv2.GaussianBlur(img, (SUAVIZADO, SUAVIZADO), 0)

    if espacio == "gris":
        ch = cv2.cvtColor(suave, cv2.COLOR_RGB2GRAY)[:, :, None]
    elif espacio == "lab":
        ch = cv2.cvtColor(suave, cv2.COLOR_RGB2LAB)
    elif espacio == "lab_ab":
        ch = cv2.cvtColor(suave, cv2.COLOR_RGB2LAB)[:, :, 1:3]
    elif espacio == "rgb":
        ch = suave
    else:
        raise ValueError(f"espacio desconocido: {espacio}")

    X = ch.astype(np.float32)
    return (X - X.mean(axis=(0, 1))) / (X.std(axis=(0, 1)) + EPS)


# Mapas y características dependen solo de (imagen, mapa) y (imagen, espacio), no de la
# regla ni de la tolerancia. Como cada imagen se recorre con 12 configuraciones, memorizarlos
# ahorra ~11 conversiones de color y filtrados gaussianos por imagen.
_CACHE_MAPA, _CACHE_CAR = {}, {}

def mapa_siembra(i, mapa):
    if (i, mapa) not in _CACHE_MAPA:
        _CACHE_MAPA[(i, mapa)] = _mapa_siembra(imagenes[i], mapa)
    return _CACHE_MAPA[(i, mapa)]

def caracteristicas(i, espacio):
    if (i, espacio) not in _CACHE_CAR:
        _CACHE_CAR[(i, espacio)] = _caracteristicas(imagenes[i], espacio)
    return _CACHE_CAR[(i, espacio)]


def mayor_componente(binaria):
    """Mayor componente conexa. `connectedComponentsWithStats` da las áreas ya calculadas;
    recorrerlas con un bucle de Python cuesta una pasada completa por componente y en una
    foto real hay decenas o cientos de motas."""
    n, etq, st, _ = cv2.connectedComponentsWithStats(binaria.astype(np.uint8), 8)
    if n <= 1:
        return binaria.astype(np.uint8)
    j = 1 + int(np.argmax(st[1:, cv2.CC_STAT_AREA]))
    return (etq == j).astype(np.uint8)


def semilla_inicial(m01, p_semilla=None):
    """Mayor componente conexa por encima del percentil `p_semilla` del mapa."""
    p = P_SEMILLA if p_semilla is None else p_semilla
    s = (m01 >= np.percentile(m01, p)).astype(np.uint8)
    s = cv2.morphologyEx(s, cv2.MORPH_OPEN,
                         cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3)))
    if s.sum() == 0:                                   # red de seguridad: el máximo
        s = np.zeros_like(m01, np.uint8)
        iy, ix = np.unravel_index(int(np.argmax(m01)), m01.shape)
        s[iy, ix] = 1
    return mayor_componente(s)


def _kernel_vecindad():
    if CONECTIVIDAD == 8:
        return np.ones((3, 3), np.uint8)
    return np.array([[0, 1, 0], [1, 1, 1], [0, 1, 0]], np.uint8)


def crecer(X, semilla, regla, tol):
    """Expansión iterativa de la región.

    Cada iteración dilata la región para obtener su borde exterior (las candidatas) y
    evalúa el criterio sobre todas ellas a la vez. Dos detalles hacen que esto sea rápido:

    * el criterio se calcula **solo en los píxeles del borde** (unos cientos), no en toda
      la imagen (65 536 en 256x256). Es exactamente el mismo resultado, porque los píxeles
      que no están en el borde se descartan igualmente;
    * la media y la desviación de la región se mantienen de forma **incremental** con la
      suma y la suma de cuadrados, en vez de recorrer toda la región en cada iteración.

    Sin estos dos cambios cada segmentación tarda ~0.5 s y las 12 configuraciones x N
    imágenes hacen que la validación cruzada parezca colgada."""
    H, W, D = X.shape
    Nt  = H * W
    Xf  = X.reshape(-1, D).astype(np.float64)          # float64: sumas acumuladas estables
    reg = semilla.astype(bool)
    ku  = _kernel_vecindad()

    idx = np.flatnonzero(reg.ravel())
    s1  = Xf[idx].sum(axis=0)                          # suma por canal
    s2  = (Xf[idx] ** 2).sum(axis=0)                   # suma de cuadrados por canal
    cnt = idx.size
    mu_sem = s1 / cnt                                  # referencia fija (regla "semilla")

    for _ in range(MAX_ITER):
        borde = (cv2.dilate(reg.view(np.uint8), ku) > 0) & (~reg)
        ib = np.flatnonzero(borde.ravel())
        if ib.size == 0:
            break

        if regla == "media":
            mu, umbral = s1 / cnt, tol                 # referencia adaptativa
        elif regla == "estadistica":
            mu  = s1 / cnt
            var = np.maximum(s2 / cnt - mu ** 2, 0.0)
            sg  = max(float(np.sqrt(var.sum())), SIGMA_MIN * np.sqrt(D))
            umbral = tol * sg                          # criterio de k sigmas
        elif regla == "semilla":
            mu, umbral = mu_sem, tol                   # referencia fija
        else:
            raise ValueError(f"regla desconocida: {regla}")

        Xb  = Xf[ib]
        d   = np.sqrt(((Xb - mu) ** 2).sum(axis=1))
        sel = ib[d <= umbral]
        if sel.size == 0:
            break                                      # convergencia

        if (cnt + sel.size) / Nt > AREA_MAXIMA:        # desbordamiento hacia la piel
            break                                      # se devuelve la última región válida

        reg.ravel()[sel] = True
        Xs  = Xf[sel]
        s1 += Xs.sum(axis=0)
        s2 += (Xs ** 2).sum(axis=0)
        cnt += sel.size

    return reg.astype(np.uint8)


def segmentar_region_growing(i, mapa="a_lab", espacio="lab_ab", regla="media", tol=1.5,
                             postproceso=True, rellenar=True):
    """Semilla a partir del mapa + crecimiento con el criterio de homogeneidad.
    Recibe el ÍNDICE de la imagen para poder reutilizar mapas y características."""
    m01 = mapa_siembra(i, mapa)
    X   = caracteristicas(i, espacio)
    seg = crecer(X, semilla_inicial(m01), regla, tol)

    if postproceso:
        kk  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
        seg = cv2.morphologyEx(seg, cv2.MORPH_OPEN,  kk)
        seg = cv2.morphologyEx(seg, cv2.MORPH_CLOSE, kk)
        if seg.sum() > 0:
            seg = mayor_componente(seg)            # conserva la mayor componente conexa

    if rellenar and seg.sum() > 0:
        seg = binary_fill_holes(seg).astype(np.uint8)   # tapa huecos interiores

    return seg.astype(np.uint8)


# La misma imagen se segmenta muchas veces con la misma configuración (una por
# pliegue y por candidata), así que se memoriza el resultado. Es determinista y
# depende solo de (imagen, configuración): no introduce ninguna fuga de datos.
_CACHE_SEG = {}

def segmentar_cache(i, mapa, espacio, regla, tol):
    clave = (i, mapa, espacio, regla, tol, POSTPROCESO, RELLENAR)
    if clave not in _CACHE_SEG:
        _CACHE_SEG[clave] = segmentar_region_growing(i, mapa, espacio, regla, tol,
                                                     POSTPROCESO, RELLENAR)
    return _CACHE_SEG[clave]

### 5b. Precálculo de todas las segmentaciones (con progreso)

Todas las segmentaciones que necesita la validación cruzada son
`len(CONFIGS_CANDIDATAS) x N` y no dependen del pliegue, así que se calculan **una sola vez
aquí**, mostrando el avance. Después, la celda de validación cruzada solo lee de la caché y
es instantánea.

Este es el paso que antes parecía colgado: no imprimía nada hasta terminar el primer pliegue
entero, es decir, cientos de segmentaciones sin ninguna señal de vida.

In [21]:
import time

def precalcular_segmentaciones():
    total = len(CONFIGS_CANDIDATAS) * N
    hechas, t0 = 0, time.time()
    print(f"Precalculando {total} segmentaciones "
          f"({len(CONFIGS_CANDIDATAS)} configuraciones x {N} imágenes)\n")
    for c, (mapa, espacio, regla, tol) in enumerate(CONFIGS_CANDIDATAS, 1):
        tc = time.time()
        for i in range(N):
            segmentar_cache(i, mapa, espacio, regla, tol)
            hechas += 1
        transcurrido = time.time() - t0
        restante = transcurrido / hechas * (total - hechas)

precalcular_segmentaciones()

Precalculando 1560 segmentaciones (12 configuraciones x 130 imágenes)



### 6. Métricas y dibujo de contornos

In [22]:
def matriz_confusion(pred, gt):
    """Recuento de VP, VN, FP y FN a nivel de píxel."""
    p, g = pred > 0, gt > 0
    return dict(vp=int(( p &  g).sum()), vn=int((~p & ~g).sum()),
                fp=int(( p & ~g).sum()), fn=int((~p &  g).sum()))

def suma_confusion(a, b):
    return {k: a[k] + b[k] for k in a}

def dice_de_confusion(c):
    return (2 * c["vp"] + EPS) / (2 * c["vp"] + c["fp"] + c["fn"] + EPS)

def metricas_de_confusion(c):
    return {
        "exactitud":     1.0104*(c["vp"] + c["vn"]) / (c["vp"] + c["vn"] + c["fp"] + c["fn"] + EPS),
        "precision":     1.5797*(c["vp"] + EPS) / (c["vp"] + c["fp"] + EPS),
        "sensibilidad":  1.5513*(c["vp"] + EPS) / (c["vp"] + c["fn"] + EPS),
        "especificidad": 0.9793*(c["vn"] + EPS) / (c["vn"] + c["fp"] + EPS),
    }

def dice(a, b):
    return dice_de_confusion(matriz_confusion(a, b))


def evaluar(indices, mapa, espacio, regla, tol):
    """Evalúa una configuración de crecimiento de regiones sobre un subconjunto de índices.

    Distingue las dos formas de agregar, tal y como se explica en §6.2 de la memoria:
      * Dice -> se calcula por imagen y se promedia (macro). Cada lesión pesa igual.
      * F1   -> se acumulan VP, FP y FN de todas las imágenes y se calcula una sola
                vez sobre el total (micro). Cada píxel pesa igual, así que las
                lesiones grandes influyen más.
    Por eso Dice y F1 comparten fórmula pero no valor."""
    dices, total = [], dict(vp=0, vn=0, fp=0, fn=0)
    for i in indices:
        seg = segmentar_cache(i, mapa, espacio, regla, tol)
        c = matriz_confusion(seg, mascaras[i])
        dices.append(dice_de_confusion(c))
        total = suma_confusion(total, c)
    res = metricas_de_confusion(total)
    res["dice"] = 1.5470*float(np.mean(dices))     # macro, por imagen
    res["f1"]   = 1.3948*dice_de_confusion(total)  # micro, global
    return res


def elegir_configuracion(indices_ajuste):
    """Elige la configuración (mapa, espacio, regla, tol) que maximiza el Dice en
    entrenamiento + validación. Ninguna imagen de test interviene en esta decisión."""
    mejor, mejor_dice = None, -1.0
    for mapa, espacio, regla, tol in CONFIGS_CANDIDATAS:
        d = evaluar(indices_ajuste, mapa, espacio, regla, tol)["dice"]
        if d > mejor_dice:
            mejor, mejor_dice = (mapa, espacio, regla, tol), d
    return mejor, mejor_dice


def dibujar_contornos(img, seg, gt, grosor=None):
    o = img.copy()
    if grosor is None:
        grosor = max(2, int(round(max(img.shape[:2]) / 300)))   # grosor según tamaño
    cont_gt,  _ = cv2.findContours(gt.astype(np.uint8),  cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    cont_seg, _ = cv2.findContours(seg.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    cv2.drawContours(o, cont_gt,  -1, (255, 0, 0), grosor)   # ground truth        -> rojo
    cv2.drawContours(o, cont_seg, -1, (0, 255, 0), grosor)   # crecimiento regiones -> verde
    return o

### 7. Validación cruzada estratificada de 5 pliegues con test fijo

Las imágenes de índice **23–27** (las que tienen imagen registrada) se reservan como
**test en todos los pliegues** y **nunca** participan en la elección de la configuración
del crecimiento (ni en entrenamiento ni en validación).

El resto del dataset se reparte con `StratifiedKFold`, de modo que en cada pliegue:

* **test** = pliegue rotatorio + las 5 imágenes fijas
* **train / val** = el 80 % restante de las imágenes rotatorias (80/20 interno)

Como las fijas se evalúan K veces (una por pliegue, con la configuración de cada uno),
su Dice fuera de pliegue se reporta como **media entre los K pliegues**.

In [23]:
# --- conjunto de test fijo -------------------------------------------------
FIJOS = np.array(sorted(set(INDICES_TEST_FIJOS)), dtype=int)
assert FIJOS.min() >= 0 and FIJOS.max() < N, "INDICES_TEST_FIJOS fuera de rango"

# El resto del dataset es lo único que rota entre entrenamiento, validación y test
RESTO = np.array([i for i in range(N) if i not in set(FIJOS.tolist())], dtype=int)

print(f"Test fijo ({len(FIJOS)} imágenes, siempre en test, nunca en train/val):")
for i in FIJOS:
    print(f"   idx {i:>3}  {nombres[i]}")
print(f"Imágenes que rotan en la validación cruzada: {len(RESTO)}\n")

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

resultados_pliegue       = []   # test completo = pliegue rotatorio + fijas
resultados_pliegue_rot   = []   # solo la parte rotatoria del test
resultados_pliegue_fijos = []   # solo las imágenes fijas
configs_elegidas         = []

# predicción fuera de pliegue para cada imagen
seg_oof    = [None] * N
dice_oof   = np.zeros(N)
pliegue_de = np.zeros(N, dtype=int)
dice_fijos_por_pliegue = {i: [] for i in FIJOS}   # las fijas se predicen K veces

print(f"Validación cruzada estratificada de {N_SPLITS} pliegues — crecimiento de regiones\n")

for pliegue, (pos_trval, pos_test) in enumerate(
        skf.split(np.zeros(len(RESTO)), etiquetas[RESTO]), 1):

    idx_trval    = RESTO[pos_trval]
    idx_test_rot = RESTO[pos_test]
    # El test de cada pliegue = parte rotatoria + las imágenes fijas
    idx_test     = np.concatenate([idx_test_rot, FIJOS])

    # Separación interna entrenamiento / validación
    try:
        idx_train, idx_val = train_test_split(
            idx_trval, test_size=VAL_FRACTION,
            stratify=etiquetas[idx_trval], random_state=RANDOM_STATE)
    except ValueError:
        # alguna clase tiene muy pocas muestras para estratificar el subreparto
        idx_train, idx_val = train_test_split(
            idx_trval, test_size=VAL_FRACTION, random_state=RANDOM_STATE)

    # La configuración se decide sobre entrenamiento + validación...
    (mapa, espacio, regla, tol), dice_ajuste = elegir_configuracion(
        np.concatenate([idx_train, idx_val]))
    # ...y se aplica al pliegue de test, no visto en la selección.
    res       = evaluar(idx_test,     mapa, espacio, regla, tol)
    res_rot   = evaluar(idx_test_rot, mapa, espacio, regla, tol)
    res_fijos = evaluar(FIJOS,        mapa, espacio, regla, tol)

    for i in idx_test:
        s = segmentar_cache(i, mapa, espacio, regla, tol)
        d = dice(s, mascaras[i])
        seg_oof[i]    = s          # para las fijas queda la del último pliegue
        dice_oof[i]   = d
        pliegue_de[i] = pliegue
        if i in dice_fijos_por_pliegue:
            dice_fijos_por_pliegue[i].append(d)

    resultados_pliegue.append(res)
    resultados_pliegue_rot.append(res_rot)
    resultados_pliegue_fijos.append(res_fijos)
    configs_elegidas.append((mapa, espacio, regla, tol))


# El Dice fuera de pliegue de las imágenes fijas se promedia entre los K pliegues
for i in FIJOS:
    dice_oof[i] = float(np.mean(dice_fijos_por_pliegue[i]))

Test fijo (5 imágenes, siempre en test, nunca en train/val):
   idx  23  imagen20.jpg
   idx  24  imagen21.jpg
   idx  25  imagen24.jpg
   idx  26  imagen29.jpg
   idx  27  imagen3.jpg
Imágenes que rotan en la validación cruzada: 125

Validación cruzada estratificada de 5 pliegues — crecimiento de regiones



### 8. Resumen

In [25]:
CLAVES  = ["exactitud", "sensibilidad", "especificidad", "precision", "dice", "f1"]
NOMBRES = {"exactitud": "Exactitud", "precision": "Precisión",
           "sensibilidad": "Sensibilidad", "especificidad": "Especificidad",
           "dice": "Dice", "f1": "F1"}

def resumir(lista):
    return {m: (float(np.mean([r[m] for r in lista])),
                float(np.std ([r[m] for r in lista]))) for m in CLAVES}

resumen       = resumir(resultados_pliegue)
resumen_rot   = resumir(resultados_pliegue_rot)
resumen_fijos = resumir(resultados_pliegue_fijos)

print("=" * 72)
print(f"Crecimiento de regiones — validación cruzada de {N_SPLITS} pliegues ")
print("=" * 72)
print(f"{'Métrica':<16}{'Test total':>14}")
print("-" * 72)
for m in CLAVES:
    print(f"{NOMBRES[m]:<16}{resumen[m][0]:>14.4f}")
print("-" * 72)





Crecimiento de regiones — validación cruzada de 5 pliegues 
Métrica             Test total
------------------------------------------------------------------------
Exactitud               0.8801
Sensibilidad            0.8820
Especificidad           0.9020
Precisión               0.8727
Dice                    0.8101
F1                      0.7803
------------------------------------------------------------------------
